# 10장 (실습 1): AI 사용 정책을 코드로 집행하기

> 책 『AI 보안, 클라우드에서 시작한다』 10장 [실습] 1번을 노트북에서 단계별로 실행하는 버전입니다. 10-2에서 AI 시대의 정책은 사람이 기억하는 규칙이 아니라 **시스템이 자동으로 집행하고 기록하는 규칙**이어야 한다고 했습니다. 표 10-4의 위험 등급 기준과 표 10-5의 책임 원칙을 정책 코드(Policy as Code)로 옮기고, AI 자산 등록 파일을 검사해 기준에 맞지 않으면 배포·운영 승인을 막습니다. 소요 시간은 30~40분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. AI 자산 등록 파일을 **정책 코드(OPA Rego)** 로 검사하고, 위반이 있으면 종료 코드 1로 승인을 막습니다.
2. 등급을 사람이 고르지 않고 **데이터와 행동으로 계산**해야 하는 이유를 확인합니다.
3. 등록 파일을 고쳐 해결되는 위반과 **실제 작업이 필요한 위반**을 구분하고, 규칙과 정책 테스트를 직접 추가합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | 예정 |
| 1 | 등록 파일과 정책 살펴보기 | 예정 |
| 2 | 검사 실행 | 예정 |
| 3 | 등록 파일 고치기 | 예정 |
| 4 | 규칙 추가하기 | 예정 |
| 5 | CI에 연결하기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

## 실습 시나리오

가상 조직(직원 500명 제조기업)은 AI 자산 8개와 Agent가 쓰는 MCP Server 3개를 등록 파일 하나로 관리합니다. 지금까지는 보안 담당자가 분기마다 이 목록을 엑셀로 내려받아 눈으로 확인했습니다. 그사이 Agent의 권한이 늘었는데 등급은 그대로였고, 담당자가 바뀐 앱은 책임자 칸이 비었습니다. 이제 등록 파일이 바뀔 때마다 정책 코드가 자동으로 검사하게 합니다.

```text
 registry/ai-assets.yaml (AI 자산 등록 파일) ─┐
 policy/*.rego (정책 코드) ───────────────────┼─▶ OPA ─▶ 위반 목록 ─▶ check.sh 종료 코드 1 ─▶ CI가 병합 차단
 기준일(TODAY) ───────────────────────────────┘
```

실습의 조직, 사람, 로그는 모두 가상입니다.

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Docker가 필요합니다. 정책 엔진 OPA는 Docker 이미지(`openpolicyagent/opa:1.0.0`)로 실행하므로 따로 설치하지 않아도 됩니다. API Key는 필요 없습니다.
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch10-policy-as-code-shadow-ai` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch10-policy-as-code-shadow-ai && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch10-policy-as-code-shadow-ai` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 이 노트북은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, OPA 1.0.0에서 끝까지 실행해 확인했습니다. 실행 결과가 저장된 버전은 `ch10_lab1_policy_as_code_executed.ipynb`입니다.

> 📌 (참고) 검사의 기준일은 README와 같은 `2026-10-20`으로 고정합니다(`TODAY`). 기준일을 빼면 실행한 날로 검사하므로 [재평가] 위반이 더 나올 수 있습니다(2.2).

## 0. 실습 준비와 실습 파일

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 등록 파일과 정책 살펴보기 | 예정 |
| 2 | 검사 실행 | 예정 |
| 3 | 등록 파일 고치기 | 예정 |
| 4 | 규칙 추가하기 | 예정 |
| 5 | CI에 연결하기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "docker": "docker --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

`[표 실습10-1]` 실습 1의 구성

| 파일 | 역할 |
|---|---|
| `policy-as-code/registry/ai-assets.yaml` | AI 자산 등록 파일(허용 목록). 자산마다 소유 부서, 최종 책임자, 신고 등급, 다루는 데이터와 하는 행동(`data`), 승인과 점검 기록(`controls`)을 적음 |
| `policy-as-code/policy/ai_assets.rego` | 정책 코드. 표 10-4와 10-5를 일곱 가지 규칙으로 옮김 |
| `policy-as-code/policy/ai_assets_test.rego` | 정책 자체의 테스트. 정책을 고칠 때 기존 규칙이 깨지지 않았는지 확인 |
| `policy-as-code/check.sh` | 등록 파일을 정책으로 검사하고, 위반이 있으면 종료 코드 1을 돌려줌 |
| `policy-as-code/ci/github-actions.yml` | 등록 파일이 바뀌는 Pull Request마다 검사하는 CI 설정 예시 |

실습 파일을 `~/ch10-lab`으로 복사하고 OPA 이미지를 받아 둡니다. 3·4단계에서 등록 파일과 정책을 고치는데, 저장소의 원본은 그대로 두기 위해서입니다.

In [ ]:
TODAY = "2026-10-20"
WORK = Path.home() / "ch10-lab" / "policy-as-code"
sh(f"rm -rf ~/ch10-lab && cp -r '{LAB}' ~/ch10-lab && rm -f ~/ch10-lab/*.ipynb && chmod +x {WORK}/check.sh")
print(sh("docker pull -q openpolicyagent/opa:1.0.0"))
REG = WORK / "registry" / "ai-assets.yaml"


def check(today=TODAY):
    """check.sh를 실행해 (출력, 종료 코드)를 돌려줍니다."""
    r = subprocess.run(["./check.sh"], cwd=WORK, capture_output=True, text=True, env=dict(os.environ, TODAY=today))
    return r.stdout, r.returncode


def opa(query):
    """정책과 등록 파일을 읽은 OPA에 질문 하나를 던져 JSON 결과를 돌려줍니다."""
    out = sh(f"cd {WORK} && echo '{{\"today\":\"{TODAY}\"}}' | docker run --rm -i -v \"$PWD\":/work -w /work "
             f"openpolicyagent/opa:1.0.0 eval --stdin-input -d policy/ -d registry/ai-assets.yaml -f raw '{query}'")
    return json.loads(out)

## 1. 등록 파일과 정책 살펴보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 등록 파일과 정책 살펴보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 검사 실행 | 예정 |
| 3 | 등록 파일 고치기 | 예정 |
| 4 | 규칙 추가하기 | 예정 |
| 5 | CI에 연결하기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

`[표 실습10-2]` 정책 코드의 규칙과 10장의 기준

| 분류 | 규칙 | 10장의 기준 |
|---|---|---|
| [책임] | 소유 부서와 최종 책임자(A)가 있어야 한다 | 표 10-5 |
| [등급] | 신고 등급이 데이터와 행동으로 정해지는 기준 등급보다 낮으면 안 된다 | 표 10-4 |
| [승인] | 운영 중인 자산은 등급에 맞는 곳의 승인이 있어야 한다 (매우 높음: 보안위원회, 높음: 보안팀 이상, 중간: 부서장 이상) | 표 10-4 |
| [테스트] | 높음 이상은 보안 테스트, 매우 높음은 Agent 시나리오 테스트까지 통과 기록이 있어야 한다 | 표 10-4, 8-3 |
| [통제] | 매우 높음 등급은 사람 승인과 Kill Switch가 켜져 있어야 한다 | 표 10-3, 8-4 |
| [재평가] | 등급별 재평가 주기(90일, 180일, 365일)를 넘기지 않는다 | 표 10-4 |
| [Tool] | Agent는 승인된 MCP Server만 쓴다 | 10-3 허용 목록 |

### 1.1. 구매 자동화 Agent 살펴보기

In [ ]:
%%bash
cd ~/ch10-lab/policy-as-code
sed -n '/id: AI-004/,/^$/p' registry/ai-assets.yaml

구매 자동화 Agent는 발주를 실행하고(`executes_changes`) 메일을 보내는데(`external_send`) 등급은 "높음"으로 신고되어 있습니다. 정책 코드에서 이 판단을 하는 부분은 다음과 같습니다.

In [ ]:
%%bash
cd ~/ch10-lab/policy-as-code
sed -n '/^required_grade/,/^} else := "낮음"/p' policy/ai_assets.rego

### 1.2. 모든 자산의 신고 등급과 기준 등급

핵심은 [등급] 규칙입니다. 등급을 사람이 고르게 두면 승인 절차를 피하려고 낮게 신고하기 쉽습니다. 그래서 정책 코드는 자산이 **무엇을 다루고 무엇을 할 수 있는지**(`data`)로 기준 등급을 직접 계산합니다. OPA에 직접 물어 모든 자산의 두 등급을 나란히 봅니다.

In [ ]:
grades = opa("[[a.id, a.name, object.get(a, \"declared_grade\", \"\"), data.aigov.required_grade(a), "
             "[k | some k, v in a.data; v == true]] | some a in data.assets]")
table(["자산", "이름", "신고 등급", "기준 등급(계산)", "근거(data)"],
      [(i, n, d or "(없음)", f"**{r}**" if d != r else r, ", ".join(x) or "-") for i, n, d, r, x in grades])

외부로 데이터를 보내거나 실제 변경을 실행하면 매우 높음, 개인정보·기밀을 다루거나 인터넷에 노출되면 높음, 사내 데이터를 조회하면 중간입니다. 굵게 표시된 자산은 신고 등급이 기준 등급과 다릅니다.

## 2. 검사 실행

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 등록 파일과 정책 살펴보기 | ✅ 완료 |
| 2 | 검사 실행 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 등록 파일 고치기 | 예정 |
| 4 | 규칙 추가하기 | 예정 |
| 5 | CI에 연결하기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

In [ ]:
out, code = check()
print(out)
print("종료 코드", code)

### 2.1. 위반을 자산·분류별로 보기

위반 메시지는 "자산 ID [분류] 이름: 내용" 형식입니다. 자산과 분류로 펼쳐 봅니다.

In [ ]:
import re
from collections import Counter

CATS = ["책임", "등급", "승인", "테스트", "통제", "재평가", "Tool"]


def matrix(out):
    c = Counter(re.findall(r"^  (AI-\d+) \[(\S+)\]", out, re.M))
    assets = sorted({a for a, _ in c})
    table(["자산", *CATS, "합계"], [(a, *(c[(a, k)] or "" for k in CATS), sum(c[(a, k)] for k in CATS)) for a in assets])


matrix(out)

결과를 읽는 방법은 다음과 같습니다.

- **위반 하나가 다른 위반을 부릅니다**: AI-004는 등급이 매우 높음으로 올라가자 보안위원회 승인, Agent 시나리오 테스트, Kill Switch가 한꺼번에 요구됩니다. 등급을 낮게 신고하면 이 통제가 모두 빠지기 때문에 [등급] 규칙이 가장 중요합니다.
- **승인된 Agent도 위반할 수 있습니다**: AI-005는 모든 승인과 테스트를 갖췄지만, 승인되지 않은 MCP Server를 Tool로 쓰고 있습니다. Agent의 위험은 Agent 자체보다 Agent가 쓰는 Tool에서 늘어나는 경우가 많습니다(5-3).
- **시간이 지나면 위반이 생깁니다**: AI-002와 AI-007은 등록할 때는 문제가 없었지만 재평가 주기를 넘겼습니다.

### 2.2. 기준일을 옮기면

등록 파일을 하나도 바꾸지 않고 기준일만 옮겨 [재평가] 위반 수를 셉니다. 검사를 등록할 때 한 번이 아니라 매일 돌려야 하는 이유입니다.

In [ ]:
rows = []
for today in ["2026-10-20", "2027-01-20", "2027-04-20", "2027-10-20"]:
    o, _ = check(today)
    n = len(re.findall(r"\[재평가\]", o))
    who = sorted(set(re.findall(r"^  (AI-\d+) \[재평가\]", o, re.M)))
    rows.append((today, n, ", ".join(who)))
table(["기준일", "[재평가] 위반", "해당 자산"], rows)

## 3. 등록 파일 고치기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 등록 파일과 정책 살펴보기 | ✅ 완료 |
| 2 | 검사 실행 | ✅ 완료 |
| 3 | 등록 파일 고치기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 규칙 추가하기 | 예정 |
| 5 | CI에 연결하기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

AI-006의 담당 부서와 협의해 최종 책임자를 정하고, 등급을 기준 등급으로 올리고, 보안팀 승인을 받았다고 합시다. README에서는 편집기로 고치지만, 노트북에서는 AI-006 항목의 세 줄만 바꾸는 함수로 고칩니다.

In [ ]:
def edit_asset(asset_id, **changes):
    """등록 파일에서 자산 하나의 항목만 찾아 값을 바꿉니다 (다른 자산과 주석은 그대로 둡니다)."""
    text = REG.read_text()
    start = text.index(f"- id: {asset_id}")
    end = text.find("\n\n", start)
    block = text[start:end]
    for key, value in changes.items():
        block = re.sub(rf"({key}: )[^,}}\n]*", rf"\g<1>{value}", block, count=1)
    REG.write_text(text[:start] + block + text[end:])
    print(block)


edit_asset("AI-006", accountable="한영업", declared_grade="높음", approved_by="보안팀")

In [ ]:
out, code = check()
print("\n".join(l for l in out.splitlines() if "AI-006" in l or "위반" in l))

책임자, 등급, 승인은 문서를 고쳐서 해결했지만 [테스트] 위반은 남습니다. 이것은 등록 파일에 "통과"라고 적어서 해결할 일이 아니라 실제로 보안 테스트를 해야 하는 일입니다(8-3). 실무에서는 `security_test` 값을 사람이 적지 않고 CI의 테스트 결과가 자동으로 채우게 해서, 정책 검사가 테스트 결과를 직접 확인하도록 연결합니다.

> 🙋 (생각해 보기)
> 누군가 `security_test: 통과`를 등록 파일에 그냥 적으면 위반이 사라집니다. 이 값을 사람이 쓰지 못하게 하려면 무엇이 필요할까요? (8장 실습 1·2의 보안 Gate 결과를 파이프라인이 직접 기록)

## 4. 규칙 추가하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 등록 파일과 정책 살펴보기 | ✅ 완료 |
| 2 | 검사 실행 | ✅ 완료 |
| 3 | 등록 파일 고치기 | ✅ 완료 |
| 4 | 규칙 추가하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | CI에 연결하기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

정책은 계속 늘어납니다. 10-3에서 "정보가 나가는 길목"마다 통제를 두라고 했으므로, 외부로 데이터를 보내는 자산은 허용 목적지 목록이 있어야 한다는 규칙을 추가합니다. 같은 `package aigov`의 새 파일에 `deny` 규칙을 하나 더 쓰면, 기존 규칙과 함께 모입니다.

In [ ]:
(WORK / "policy" / "egress.rego").write_text("""package aigov

# 외부로 데이터를 보내는 자산은 허용 목적지 목록이 있어야 한다 (10-3 길목 통제)
deny contains msg if {
\tsome a in data.assets
\ta.data.external_send == true
\tcount(object.get(a, "egress_allowlist", [])) == 0
\tmsg := sprintf("%s [Egress] %s: 외부로 데이터를 보내는데 허용 목적지 목록(egress_allowlist)이 없습니다", [a.id, a.name])
}
""")
out, code = check()
print("\n".join(l for l in out.splitlines() if "Egress" in l or "위반" in l))

규칙을 추가했으면 정책 테스트로 기존 규칙이 깨지지 않았는지 확인합니다.

In [ ]:
%%bash
cd ~/ch10-lab/policy-as-code
docker run --rm -v "$PWD":/work -w /work openpolicyagent/opa:1.0.0 test policy/ -v

규칙 파일 하나를 넣었을 뿐인데 모든 자산에 같은 기준이 적용됩니다. 정책도 코드이므로 코드처럼 테스트하고, 변경 이력을 남기고, 검토를 거쳐 반영합니다.

## 5. CI에 연결하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 등록 파일과 정책 살펴보기 | ✅ 완료 |
| 2 | 검사 실행 | ✅ 완료 |
| 3 | 등록 파일 고치기 | ✅ 완료 |
| 4 | 규칙 추가하기 | ✅ 완료 |
| 5 | CI에 연결하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

`ci/github-actions.yml`은 등록 파일이나 정책이 바뀌는 Pull Request마다 정책 테스트와 검사를 실행하는 예시입니다. 이 검사를 브랜치 보호 규칙의 필수 검사로 지정하면, 위반이 있는 등록 파일은 병합되지 않습니다.

In [ ]:
%%bash
cd ~/ch10-lab/policy-as-code
cat ci/github-actions.yml
echo
TODAY=2026-10-20 ./check.sh > /dev/null || echo "검사 실패 → Pull Request 병합 차단"

이렇게 하면 "AI 자산은 등급에 맞게 승인받는다"는 정책이 사람의 기억이 아니라 병합 조건으로 집행됩니다. 예외가 필요하면 등록 파일에 예외와 만료일을 적고 그 변경 자체를 검토받게 해서, 예외도 기록으로 남깁니다.

`[표 실습10-3]` 실습 1과 실무 도구

| 이 실습 | 실무에서 같은 역할 (예) |
|---|---|
| `ai-assets.yaml` | AI 자산 레지스트리, CMDB의 AI 자산 항목(9-1), AI-SPM의 자산 목록 |
| Rego 정책과 `check.sh` | OPA·Conftest, 클라우드의 정책 서비스(AWS Config 규칙, Azure Policy), Kyverno |
| CI 필수 검사 | GitHub Actions·GitLab CI의 필수 검사, 배포 승인 단계 |
| `security_test` 값 | 8장 보안 Gate의 판정 결과를 자동으로 기록 |

## 6. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 등록 파일과 정책 살펴보기 | ✅ 완료 |
| 2 | 검사 실행 | ✅ 완료 |
| 3 | 등록 파일 고치기 | ✅ 완료 |
| 4 | 규칙 추가하기 | ✅ 완료 |
| 5 | CI에 연결하기 | ✅ 완료 |
| 6 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 정리 | 예정 |

### 6.1. 체크포인트

- [ ] AI-004가 등급을 "높음"으로 신고했을 때 빠져 있던 통제는 무엇이었나요? 등급을 사람이 고르게 두면 어떤 일이 생길까요?
- [ ] 등록 파일을 고쳐서 해결한 위반과, 실제 작업을 해야 해결되는 위반은 각각 무엇이었나요?
- [ ] 재평가 주기 위반은 등록할 때의 검사만으로 찾을 수 있었을까요? 검사는 언제 돌아야 할까요?
- [ ] 규칙을 추가할 때 정책 테스트(`opa test`)가 필요한 이유는 무엇인가요?

### 6.2. 과제 1: 규칙과 테스트 함께 추가하기

실습 2에서는 MCP Gateway 기록의 Agent ID를 등록 파일과 대조합니다. 그러려면 **Agent 자산에는 `agent_id`가 반드시 있어야** 합니다. 이 규칙과 그 테스트를 함께 씁니다.

1. `RULE`에 "`type`이 `Agent`인데 `agent_id`가 없거나 비어 있으면 [식별] 위반" 규칙을 씁니다.
2. `TEST`에 `ai_assets_test.rego`의 `agent`에서 `agent_id`를 뺀 자산이 위반되는지 확인하는 테스트를 씁니다. (참고: 테스트 파일의 `agent`에는 원래 `agent_id`가 없습니다. 먼저 넣어 주어야 기존 테스트 `test_compliant_agent_passes`가 깨지지 않습니다.)
3. 셀을 실행하면 두 파일을 만들고 `opa test`와 검사를 돌립니다. 테스트가 모두 PASS이고, 지금 등록 파일에는 [식별] 위반이 없어야 합니다.

In [ ]:
RULE = """package aigov

# TODO: Agent 자산에는 agent_id가 있어야 한다
"""

TEST = """package aigov_test

import data.aigov

# TODO: agent_id가 없는 Agent는 [식별] 위반이어야 한다
"""

(WORK / "policy" / "identity.rego").write_text(RULE)
(WORK / "policy" / "identity_test.rego").write_text(TEST)
print(sh(f"cd {WORK} && docker run --rm -v \"$PWD\":/work -w /work openpolicyagent/opa:1.0.0 test policy/ -v 2>&1"))
out, code = check()
print("[식별] 위반:", [l.strip() for l in out.splitlines() if "[식별]" in l] or "없음")

<details>
<summary>💡 힌트 보기</summary>

```python
RULE = """package aigov

deny contains msg if {
	some a in data.assets
	a.type == "Agent"
	object.get(a, "agent_id", "") in {"", null}
	msg := sprintf("%s [식별] %s: Agent인데 agent_id가 없습니다", [a.id, a.name])
}
"""

TEST = """package aigov_test

test_agent_without_id_denied if {
	a := object.remove(agent, ["agent_id"])
	some m in deny_for(a)
	contains(m, "[식별]")
}
"""
```

규칙만 넣고 `opa test`를 돌리면 기존 `test_compliant_agent_passes`가 **실패**합니다. 테스트 파일의 `agent`("모든 기준을 갖춘 Agent")에 `agent_id`가 없기 때문입니다. 새 규칙이 "기준을 갖춘 Agent"의 정의를 바꾼 것이므로, 기존 테스트의 기준도 함께 고칩니다. 위 셀을 실행하기 전에 다음 한 줄을 실행합니다.

```python
f = WORK / "policy" / "ai_assets_test.rego"
f.write_text(f.read_text().replace('"type": "Agent",', '"type": "Agent", "agent_id": "AGT-test",'))
```

정책 테스트는 이처럼 새 규칙이 기존 판단을 어떻게 바꾸는지 바로 알려 줍니다. 같은 패키지(`aigov_test`)에 둔 테스트 파일끼리는 `agent`, `deny_for`를 그대로 함께 씁니다.
</details>

### 6.3. 과제 2: 예외 처리 절차 설계 (서술)

AI-006의 [테스트] 위반은 보안 테스트 일정 때문에 한 달 뒤에야 해결됩니다. 그동안 운영을 계속하려면 예외가 필요합니다. 아래 질문에 답해 보세요.

1. 등록 파일에 예외를 적는다면 어떤 항목이 필요할까요? (예외 대상 규칙, 사유, 승인자, 만료일, 보완 통제)
2. 만료일이 지난 예외를 정책 검사가 어떻게 다뤄야 할까요?

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 7. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 등록 파일과 정책 살펴보기 | ✅ 완료 |
| 2 | 검사 실행 | ✅ 완료 |
| 3 | 등록 파일 고치기 | ✅ 완료 |
| 4 | 규칙 추가하기 | ✅ 완료 |
| 5 | CI에 연결하기 | ✅ 완료 |
| 6 | 체크포인트와 과제 | ✅ 완료 |
| 7 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

README에서는 등록 파일과 추가한 규칙을 실습 2에서 계속 씁니다. 노트북은 실습마다 처음부터 다시 복사하므로, 이 노트북의 실습 폴더는 지웁니다.

In [ ]:
print(sh("rm -rf ~/ch10-lab && echo '실습 폴더 삭제됨'"))

---
> 다음 실습 2에서는 같은 등록 파일을 허용 목록으로 삼아 로그와 대조해 Shadow AI를 찾고, 찾은 것을 등록하면 이 실습의 정책 검사로 이어지는 순환을 확인합니다.